# Test Project for Week 07 - Algorithmic Trading
### Author: Hisato Kato
### Revised: 2025/3/17

**Prediction-based Trading & Event-based Backtesting**

Implement a class that uses **event-based backtesting** to backtest the following prediction-based strategy:

* ~~Data from `http://hilpisch.com/ref_eikon_eod_data.csv`.~~  `implemented`
* ~~Select one symbol from the data set.~~  `implemented`
* Create the following features:
    * ~~log return~~ `implemented`
    * ~~direction (up or down)~~ `implemented`
    * ~~log return as 5 categories~~ `implemented`
    * ~~two SMAs (short and long window)~~ `implemented`
    * ~~difference between the SMAs~~ `implemented`
    * ~~two EWMAs (short and long window)~~ `implemented`
    * ~~difference between the EWMAs~~ `implemented`
    * ~~two rolling volatilities (short and long window)~~ `implemented`
* ~~Split the data set into training (70%) and testing data.~~ `implemented`
* ~~Normalize the training features data to have~~ `implemented`
    * ~~zero mean and~~
    * ~~standard deviation of one.~~
* ~~Normalize the test features data by the same moment values as the training data.~~ `implemented`
* ~~Create lagged features data for 5 lags.~~ `implemented`
* Train and (back-)test the following algorithms for directional (long/short) trading (from `scikit-learn`):
    * `GaussianNB()`
    * `LogisticRegression()`
    * `DecisionTreeClassifier()`
    * `SVC()`
    * `MLPClassifier()`
* Compare the performance of the different models numerically.

For the implementation, you can rely e.g. on the Python classes as presented in the PyAlgo class sessions and the resources.

## Imports and setups

In [58]:
import numpy as np
import pandas as pd
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score
from sklearn import preprocessing

## The `BacktestBase` class definition copied from the course
#### Changed the data URL from 'http://hilpisch.com/pyalgo_eikon_eod_data.csv' to
#### 'http://hilpisch.com/ref_eikon_eod_data.csv'

In [2]:
#
# Python Script with Base Class
# for Event-based Backtesting
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#

class BacktestBase(object):
    ''' Base class for event-based backtesting of trading strategies.

    Attributes
    ==========
    symbol: str
        TR RIC (financial instrument) to be used
    start: str
        start date for data selection
    end: str
        end date for data selection
    amount: float
        amount to be invested either once or per trade
    ftc: float
        fixed transaction costs per trade (buy or sell)
    ptc: float
        proportional transaction costs per trade (buy or sell)

    Methods
    =======
    get_data:
        retrieves and prepares the base data set
    plot_data:
        plots the closing price for the symbol
    get_date_price:
        returns the date and price for the given bar
    print_balance:
        prints out the current (cash) balance
    print_net_wealth:
        prints auf the current net wealth
    place_buy_order:
        places a buy order
    place_sell_order:
        places a sell order
    close_out:
        closes out a long or short position
    '''

    def __init__(self, symbol, start, end, amount,
                 ftc=0.0, ptc=0.0, verbose=True):
        self.symbol = symbol
        self.start = start
        self.end = end
        self.initial_amount = amount
        self.amount = amount
        self.ftc = ftc
        self.ptc = ptc
        self.units = 0
        self.position = 0
        self.trades = 0
        self.verbose = verbose
        self.get_data()

    def get_data(self):
        ''' Retrieves and prepares the data.
        '''
        raw = pd.read_csv('http://hilpisch.com/ref_eikon_eod_data.csv', # CHANGED
                          index_col=0, parse_dates=True).dropna()
        raw = pd.DataFrame(raw[self.symbol])
        raw = raw.loc[self.start:self.end]
        raw.rename(columns={self.symbol: 'price'}, inplace=True)
        raw['return'] = np.log(raw / raw.shift(1))
        self.data = raw.dropna()

    def plot_data(self, cols=None):
        ''' Plots the closing prices for symbol.
        '''
        if cols is None:
            cols = ['price']
        self.data['price'].plot(figsize=(10, 6), title=self.symbol)

    def get_date_price(self, bar):
        ''' Return date and price for bar.
        '''
        date = str(self.data.index[bar])[:10]
        price = self.data.price.iloc[bar]
        return date, price

    def print_balance(self, bar):
        ''' Print out current cash balance info.
        '''
        date, price = self.get_date_price(bar)
        print(f'{date} | current balance {self.amount:.2f}')

    def print_net_wealth(self, bar):
        ''' Print out current cash balance info.
        '''
        date, price = self.get_date_price(bar)
        net_wealth = self.units * price + self.amount
        print(f'{date} | current net wealth {net_wealth:.2f}')

    def place_buy_order(self, bar, units=None, amount=None):
        ''' Place a buy order.
        '''
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / price)
        self.amount -= (units * price) * (1 + self.ptc) + self.ftc
        self.units += units
        self.trades += 1
        if self.verbose:
            print(f'{date} | selling {units} units at {price:.2f}')
            self.print_balance(bar)
            self.print_net_wealth(bar)

    def place_sell_order(self, bar, units=None, amount=None):
        ''' Place a sell order.
        '''
        date, price = self.get_date_price(bar)
        if units is None:
            units = int(amount / price)
        self.amount += (units * price) * (1 - self.ptc) - self.ftc
        self.units -= units
        self.trades += 1
        if self.verbose:
            print(f'{date} | selling {units} units at {price:.2f}')
            self.print_balance(bar)
            self.print_net_wealth(bar)

    def close_out(self, bar):
        ''' Closing out a long or short position.
        '''
        date, price = self.get_date_price(bar)
        self.amount += self.units * price
        if self.verbose:
            print(f'{date} | inventory {self.units} units at {price:.2f}')
            print('=' * 57)
        print('Final balance   [$] {:.2f}'.format(self.amount))
        perf = ((self.amount - self.initial_amount) /
                self.initial_amount * 100)
        print('Net Performance [%] {:.2f}'.format(perf))
        print('=' * 57)

#if __name__ == '__main__':
#    bb = BacktestBase('AAPL.O', '2010-1-1', '2019-12-31', 10000)
#    print(bb.data.info())
#    print(bb.data.tail())
#    bb.plot_data()
    # plt.savefig('../../images/ch06/backtestbaseplot.png')


## Defining an event-based backtesting class

In [80]:
class EventBasedBackTesting(BacktestBase):
    def __init__(self, symbol, start, end, amount,
                 ftc=0.0, ptc=0.0, verbose=True):
        super(EventBasedBackTesting, self).__init__(symbol, start, end, amount,
                 ftc, ptc, verbose)
        self._prepare_features()
        
    def _prepare_features(self):
        # Direction (up or down): pff2e p.495
        self.data['direction'] = np.sign(self.data['return']).astype(int) 
        
        # Five Binary Features: pff2e p.506
        self.create_lags(self.data)
        
        # Five Digitized Features: pff2e pp.508
        mu = self.data['return'].mean()
        v = self.data['return'].std()
        bins = [mu-v, mu, mu+v]
        self.create_bins(self.data, bins)

        # Two SMAs: pff2e p.486
        window1 = 42
        window2 = 252
        self.data['SMA1'] = self.data['price'].rolling(window1).mean()
        self.data['SMA2'] = self.data['price'].rolling(window2).mean()
        

        # difference between the SMAs
        self.data['diffSMA'] = self.data['SMA1'] - self.data['SMA2']

        # Two EWMAs: Ref. Tutorial 04
        self.data['EWMA1'] = self.data['price'].ewm(halflife=window1).mean()
        self.data['EWMA2'] = self.data['price'].ewm(halflife=window2).mean()

        # difference between the EWMAs
        self.data['diffEWMA'] = self.data['EWMA1'] - self.data['EWMA2']

        # Two rolling volatilities: pff2e p.218
        self.data['Volat1'] = self.data['price'].rolling(window1).std()
        self.data['Volat2'] = self.data['price'].rolling(window2).std()

        # Drop NaN rows for SMA, EWMA, and Volatility
        self.data.dropna(inplace=True)

    def create_lags(self, data): # pff2e pp.496
        lags = 5
        global cols # referenced in create_bins
        cols = []
        for lag in range(1, lags+1):
            col = 'lags_{}'.format(lag)
            data[col] = data['return'].shift(lag)
            cols.append(col)
        data.dropna(inplace=True)

    def create_bins(self, data, bins=[0]): # pff2e p.502
        global cols_bin
        cols_bin = []
        for col in cols: # references cols in create_lags
            col_bin = col + '_bin'
            data[col_bin] = np.digitize(data[col], bins=bins)
            cols_bin.append(col_bin)

    def train_test_split(self):
        # Train-test split
        split_loc = int(len(self.data) * 0.7)
        self.train = self.data[:split_loc].copy()
        self.test =  self.data[split_loc:].copy() 
        
    def normalize_features(self):
        # Features to be used for fit and predict
        # 1 to 5. cols_bin: log-return-based, digitized, not to be normalized
        # 6. diffSMA: to be normalized
        # 7. diffEWMA: to be normalized
        # 8. Volat1: to be normalized
        # 9. Volat2: to be normalized
        
        diffSMA_mean = self.train['diffSMA'].mean()
        diffSMA_std  = self.train['diffSMA'].std()
        self.train['diffSMA'] = (self.train['diffSMA'] - diffSMA_mean) / diffSMA_std
        self.test['diffSMA']  = (self.test['diffSMA'] - diffSMA_mean) / diffSMA_std

        diffEWMA_mean = self.train['diffEWMA'].mean()
        diffEWMA_std  = self.train['diffEWMA'].std()
        self.train['diffEWMA'] = (self.train['diffEWMA'] - diffEWMA_mean) / diffEWMA_std
        self.test['diffEWMA']  = (self.test['diffEWMA'] - diffEWMA_mean) / diffEWMA_std

        Volat1_mean = self.train['Volat1'].mean()
        Volat1_std  = self.train['Volat1'].std()
        self.train['Volat1'] = (self.train['Volat1'] - Volat1_mean) / Volat1_std
        self.test['Volat1'] = (self.test['Volat1'] - Volat1_mean) / Volat1_std

        Volat2_mean = self.train['Volat2'].mean()
        Volat2_std  = self.train['Volat2'].std()
        self.train['Volat2'] = (self.train['Volat2'] - Volat2_mean) / Volat2_std
        self.test['Volat2'] = (self.test['Volat2'] - Volat2_mean) / Volat2_std

    def train_and_backtest(self):
        # Train and backtest
        C = 1
        self.models = {
            'gauss_nb': GaussianNB(),
            'log_reg': LogisticRegression(C=C),
            'tree': DecisionTreeClassifier(),
            'svm': SVC(C=C),
            'mlp': MLPClassifier(max_iter=2000,hidden_layer_sizes=[100,100])
        }
        self.fit_models()
        self.derive_positions()

    def fit_models(self):
        self.ml_cols = cols_bin + ['diffSMA', 'diffEWMA', 'Volat1', 'Volat2']
        mfit = {model: self.models[model].fit(train[self.ml_cols], train['direction'])
                    for model in self.models.keys()}

    def derive_positions(self):
        for model in self.models.keys():
            self.test['pos_' + model] = self.models[model].predict(self.test[self.ml_cols])

## Running the event-based backtesting class

In [81]:
# Instantiate
ebbt = EventBasedBackTesting('GS.N', '2010-1-1', '2019-12-31', 10000) # Symbol GS.N = Goldman Sachs
# Train-test split
ebbt.train_test_split()

In [65]:
print(ebbt.train.head())

             price    return  direction    lags_1    lags_2    lags_3  \
Date                                                                    
2011-01-10  169.76 -0.005463         -1 -0.008866 -0.010341  0.005301   
2011-01-11  169.36 -0.002359         -1 -0.005463 -0.008866 -0.010341   
2011-01-12  171.67  0.013547          1 -0.002359 -0.005463 -0.008866   
2011-01-13  171.57 -0.000583         -1  0.013547 -0.002359 -0.005463   
2011-01-14  175.00  0.019795          1 -0.000583  0.013547 -0.002359   

              lags_4    lags_5  lags_1_bin  lags_2_bin  ...  lags_4_bin  \
Date                                                    ...               
2011-01-10  0.000173  0.028665           1           1  ...           2   
2011-01-11  0.005301  0.000173           1           1  ...           2   
2011-01-12 -0.010341  0.005301           1           1  ...           1   
2011-01-13 -0.008866 -0.010341           2           1  ...           1   
2011-01-14 -0.005463 -0.008866        

In [66]:
print(ebbt.test.tail())

             price    return  direction    lags_1    lags_2    lags_3  \
Date                                                                    
2019-06-25  196.06 -0.007267         -1  0.007879  0.001226  0.000307   
2019-06-26  197.01  0.004834          1 -0.007267  0.007879  0.001226   
2019-06-27  199.32  0.011657          1  0.004834 -0.007267  0.007879   
2019-06-28  204.60  0.026145          1  0.011657  0.004834 -0.007267   
2019-07-01  206.86  0.010985          1  0.026145  0.011657  0.004834   

              lags_4    lags_5  lags_1_bin  lags_2_bin  ...  lags_4_bin  \
Date                                                    ...               
2019-06-25  0.003379  0.021514           2           2  ...           2   
2019-06-26  0.000307  0.003379           1           2  ...           2   
2019-06-27  0.001226  0.000307           2           1  ...           2   
2019-06-28  0.007879  0.001226           2           2  ...           2   
2019-07-01 -0.007267  0.007879        

In [82]:
ebbt.normalize_features()

In [68]:
print(ebbt.train.head())

             price    return  direction    lags_1    lags_2    lags_3  \
Date                                                                    
2011-01-10  169.76 -0.005463         -1 -0.008866 -0.010341  0.005301   
2011-01-11  169.36 -0.002359         -1 -0.005463 -0.008866 -0.010341   
2011-01-12  171.67  0.013547          1 -0.002359 -0.005463 -0.008866   
2011-01-13  171.57 -0.000583         -1  0.013547 -0.002359 -0.005463   
2011-01-14  175.00  0.019795          1 -0.000583  0.013547 -0.002359   

              lags_4    lags_5  lags_1_bin  lags_2_bin  ...  lags_4_bin  \
Date                                                    ...               
2011-01-10  0.000173  0.028665           1           1  ...           2   
2011-01-11  0.005301  0.000173           1           1  ...           2   
2011-01-12 -0.010341  0.005301           1           1  ...           1   
2011-01-13 -0.008866 -0.010341           2           1  ...           1   
2011-01-14 -0.005463 -0.008866        

In [69]:
print(ebbt.test.tail())

             price    return  direction    lags_1    lags_2    lags_3  \
Date                                                                    
2019-06-25  196.06 -0.007267         -1  0.007879  0.001226  0.000307   
2019-06-26  197.01  0.004834          1 -0.007267  0.007879  0.001226   
2019-06-27  199.32  0.011657          1  0.004834 -0.007267  0.007879   
2019-06-28  204.60  0.026145          1  0.011657  0.004834 -0.007267   
2019-07-01  206.86  0.010985          1  0.026145  0.011657  0.004834   

              lags_4    lags_5  lags_1_bin  lags_2_bin  ...  lags_4_bin  \
Date                                                    ...               
2019-06-25  0.003379  0.021514           2           2  ...           2   
2019-06-26  0.000307  0.003379           1           2  ...           2   
2019-06-27  0.001226  0.000307           2           1  ...           2   
2019-06-28  0.007879  0.001226           2           2  ...           2   
2019-07-01 -0.007267  0.007879        

In [83]:
ebbt.train_and_backtest()

In [78]:
ebbt.test.head()

,price,return,direction,lags_1,lags_2,lags_3,lags_4,lags_5,lags_1_bin,lags_2_bin,...,EWMA1,EWMA2,diffEWMA,Volat1,Volat2,pos_gauss_nb,pos_log_reg,pos_tree,pos_svm,pos_mlp
Date,,,,,,,,,,,,,,,,,,,,,
2016-12-13,238.55,0.005802,1,-0.019541,0.001655,0.024697,0.017904,0.012306,0,2,...,183.654260,172.191601,0.413736,6.882686,1.113643,-1,1,-1,1,1
2016-12-14,239.93,0.005768,1,0.005802,-0.019541,0.001655,0.024697,0.017904,2,0,...,184.575385,172.379210,0.457428,7.001050,1.214614,-1,1,-1,1,-1
2016-12-15,243.00,0.012714,1,0.005768,0.005802,-0.019541,0.001655,0.024697,2,2,...,185.531683,172.574797,0.502739,7.098941,1.318467,-1,1,-1,1,-1
2016-12-16,238.90,-0.017016,-1,0.012714,0.005768,0.005802,-0.019541,0.001655,2,2,...,186.405220,172.758483,0.543830,7.137240,1.410420,-1,1,-1,1,-1
2016-12-19,239.07,0.000711,1,-0.017016,0.012714,0.005768,0.005802,-0.019541,0,2,...,187.267240,172.942127,0.584237,7.161201,1.503405,-1,1,-1,1,1


In [79]:
MLPClassifier?

Init signature:
MLPClassifier(
    hidden_layer_sizes=(100,),
    activation='relu',
    *,
    solver='adam',
    alpha=0.0001,
    batch_size='auto',
    learning_rate='constant',
    learning_rate_init=0.001,
    power_t=0.5,
    max_iter=200,
    shuffle=True,
    random_state=None,
    tol=0.0001,
    verbose=False,
    warm_start=False,
    momentum=0.9,
    nesterovs_momentum=True,
    early_stopping=False,
    validation_fraction=0.1,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-08,
    n_iter_no_change=10,
    max_fun=15000,
)
Docstring:     
Multi-layer Perceptron classifier.

This model optimizes the log-loss function using LBFGS or stochastic
gradient descent.

.. versionadded:: 0.18

Parameters
----------
hidden_layer_sizes : array-like of shape(n_layers - 2,), default=(100,)
    The ith element represents the number of neurons in the ith
    hidden layer.

activation : {'identity', 'logistic', 'tanh', 'relu'}, default='relu'
    Activation function for the hidden laye